# 5. Effet de l'industrie sur les nappes phréatiques

Objectif : déterminer si les nappes situées près des installations industrielles ont évolué plus défavorablement que les autres entre 2015 et 2025.

Démarche :
1. Mesurer l'évolution de chaque nappe entre le début et la fin de la période.
2. Mesurer la pression industrielle autour de chaque station (nombre d'usines).
3. Comparer les deux, puis chercher ce qui distingue les stations (géologie, profondeur).
4. Vérifier avec les volumes d'eau réellement pompés (BNPE).

**Étape 5 sur 5.** Ce notebook exécute `eau_01_donnees.ipynb`, puis mesure l'effet de l'industrie sur l'évolution des nappes.

Variables produites : `evo`, `annuel`, `st`, `usines`, `prelevements`, `sout`, `focus`, `tableau_corr`.

**Organisation des notebooks.**

| Notebook | Contenu | Exécute d'abord |
|---|---|---|
| `eau_01_donnees.ipynb` | Récupération et préparation des données | — |
| `eau_02_usines_stations.ipynb` | Usines et stations : premier regard | 01 |
| `eau_03_geographie.ipynb` | Altitude, roche, profondeur | 01 |
| `eau_04_climat.ipynb` | Retrait de l'effet de la pluie | 03 |
| `eau_05_industrie.ipynb` | Effet de l'industrie | 01 |
| `rapport.ipynb` | Figures et interprétations uniquement | 02, 04, 05 |

Chaque notebook exécute ceux dont il a besoin avec `%run`, sans afficher leurs sorties et une seule fois par noyau : pour tout recalculer après une modification, redémarrer le noyau.

In [ ]:
%%capture
#Étapes précédentes : exécutées sans afficher leurs figures ni leurs messages, et une seule fois par noyau
#(si les données ne sont pas encore en cache, exécuter d'abord eau_01_donnees.ipynb seul pour suivre le téléchargement)
AFFICHER = globals().get("AFFICHER", True)   #True si ce notebook est ouvert seul, False s'il est appelé par un autre notebook
_afficher_etape_05, AFFICHER = AFFICHER, False
if "ETAPE_01" not in globals():
    %run ./eau_01_donnees.ipynb
AFFICHER = _afficher_etape_05
plt.close("all")

## 5.1 Mesure de l'évolution des nappes

Pour chaque station, on compare le niveau moyen de 2015-2017 à celui de 2023-2025.

On travaille sur des moyennes annuelles : chaque année contient un hiver et un été, ce qui évite de comparer une valeur d'été à une valeur d'hiver (qui sont toujours différentes à cause de la saisonnalité).

In [ ]:
#On crée un tableau qui va contenir l'évolution (variation moyenne debut et moyenne fin) de chaque station.
mensuel = chroniques_niveau_mensuel.copy()
mensuel["annee"] = mensuel["date_mesure"].dt.year
annuel = mensuel.groupby(["code_bss", "annee"]).agg(niveau=("niveau_nappe_eau", "mean"), nb_mois=("niveau_nappe_eau", "size")).reset_index()
annuel = annuel[annuel["nb_mois"] >= 10]

debut = annuel[annuel["annee"].between(2015, 2017)].groupby("code_bss")["niveau"].agg(["mean", "size"])
fin = annuel[annuel["annee"].between(2023, 2025)].groupby("code_bss")["niveau"].agg(["mean", "size"])

evo = debut.join(fin, lsuffix="_debut", rsuffix="_fin", how="inner")
evo = evo[(evo["size_debut"] >= 2) & (evo["size_fin"] >= 2)]
evo["evolution_m"] = evo["mean_fin"] - evo["mean_debut"]

print("Stations avec une évolution calculable :", len(evo))

In [ ]:
# On crée le tableau de travail avec les données sur les stations (profondeur, géologie, evolution)
st = stations.merge(evo[["mean_debut", "evolution_m"]].reset_index(), on="code_bss")

st["profondeur_investigation"] = pd.to_numeric(st["profondeur_investigation"], errors="coerce")
st["classe_profondeur"] = pd.cut(st["profondeur_investigation"], bins=[-np.inf, 30, 80, np.inf], labels=["Superficielle (<30m)", "Intermédiaire (30-80m)", "Profonde (>80m)"])

noms = st["noms_masse_eau_edl"].astype(str)
st["famille_geologique"] = np.select(
    [noms.str.contains("Craie", case=False), noms.str.contains("Sable|Landénien", case=False), noms.str.contains("Calcaire", case=False)],
    ["Craie", "Sable", "Calcaire"],
    default="Autre"
)

print(st["evolution_m"].describe().round(2))

In [ ]:
def fig_distribution_evolutions():
    """Histogramme et classement des évolutions 2015-2017 -> 2023-2025 de chaque piézomètre."""
    med = st["evolution_m"].median()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(st["evolution_m"], bins=40, color="steelblue", edgecolor="white")
    axes[0].axvline(0, color="black", linewidth=1)
    axes[0].axvline(med, color="red", linestyle="--", label=f"Médiane : {med:+.2f} m")
    axes[0].set_xlabel("Évolution du niveau (m) : moyenne 2023-2025 − moyenne 2015-2017")
    axes[0].set_ylabel("Nombre de piézomètres")
    axes[0].set_title(f"Distribution des évolutions ({len(st)} piézomètres)")
    axes[0].legend()

    tri = st["evolution_m"].sort_values().reset_index(drop=True)
    axes[1].bar(tri.index, tri, width=1, color=np.where(tri < 0, "tab:red", "tab:blue"))
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].axhline(med, color="grey", linestyle="--", label="Médiane")
    axes[1].set_xticks([])
    axes[1].set_xlabel("Piézomètres (triés)")
    axes[1].set_ylabel("Évolution (m)")
    axes[1].set_title(f"{(tri > 0).sum()} en hausse / {(tri < 0).sum()} en baisse")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_distribution_evolutions()

**Interprétation.** Sur les 50 piézomètres étudiés, la médiane de l'évolution est de +0,06 m : la nappe typique du Nord est au même niveau en 2023-2025 qu'en 2015-2017. La grande majorité des stations varie de moins de 50 cm, et la répartition entre hausses (28) et baisses (22) est presque équilibrée. Il n'y a donc pas de baisse généralisée des nappes dans le département.

Quelques stations s'écartent nettement : trois fortes baisses (Bondues −3,57 m, Nieppe −1,76 m, Villers-en-Cauchies −1,16 m) et une forte hausse (Limont-Fontaine +2,72 m).

In [ ]:
#Écart de chaque année au niveau moyen 2015-2017 de sa station
annuel_ecart = annuel.merge(evo[["mean_debut"]], left_on="code_bss", right_index=True)
annuel_ecart["ecart"] = annuel_ecart["niveau"] - annuel_ecart["mean_debut"]

def fig_plus_fortes_evolutions():
    """Trajectoire annuelle des 5 plus fortes baisses et des 5 plus fortes hausses."""
    tri_st = st.sort_values("evolution_m")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, codes, titre in [(axes[0], tri_st["code_bss"].head(5), "Plus fortes baisses"), (axes[1], tri_st["code_bss"].tail(5), "Plus fortes hausses")]:
        for code in codes:
            serie = annuel_ecart[annuel_ecart["code_bss"] == code]
            evol = st.loc[st["code_bss"] == code, "evolution_m"].iloc[0]
            ax.plot(serie["annee"], serie["ecart"], marker="o", label=f"{code} ({evol:+.2f} m)")
        ax.axvspan(2015, 2017, color="grey", alpha=0.15)
        ax.axvspan(2023, 2025, color="grey", alpha=0.15)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(titre)
        ax.set_xlabel("Année")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Écart au niveau moyen 2015-2017 (m)")
    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_plus_fortes_evolutions()

**Interprétation.** Presque toutes les stations suivent un mouvement commun : une baisse de 2017 à 2019-2020, puis une remontée en 2024. C'est la signature du climat (années sèches, puis hiver 2023-2024 très pluvieux), qui touche toutes les nappes.

Bondues (bleu) se distingue : elle descend jusqu'à −4,5 m en 2022 et remonte peu. Un autre facteur semble s'ajouter au climat pour cette station.

Plusieurs stations « en hausse » ne le sont que grâce à l'année 2024, exceptionnellement humide : l'indicateur dépend en partie du choix des périodes.

La station de Limont-Fontaine (violet) varie de plus de 11 m, bien plus que ses voisines : ses données sont atypiques et doivent être interprétées avec prudence.

## 5.2 Premier indicateur de pression : le nombre d'usines

On ne garde que l'industrie manufacturière (codes NAF 10 à 33) et l'énergie (code 35), qui sont les activités les plus consommatrices d'eau. Pour chaque station, on compte les usines situées à moins de 2, 5 et 10 km, afin de vérifier que le résultat ne dépend pas du rayon choisi.

Les stations sont ensuite réparties en trois groupes : 0 usine, 1 à 5 usines, plus de 5 usines (dans un rayon de 5 km).

In [ ]:
# On ne garde que l'industrie manufacturière et l'énergie
code_naf = pd.to_numeric(installations_indust["code_naf"].astype(str).str[:2], errors="coerce")
usines = installations_indust[code_naf.between(10, 33) | (code_naf == 35)]
print("Usines manufacturières et énergétiques retenues :", len(usines))

#On compte les usines à proximité de chaque station avec différents rayons
for rayon_km in [2, 5, 10]:
    st[f"usines_{rayon_km}km"] = st.geometry.apply(lambda g: (usines.distance(g) <= rayon_km * 1000).sum())

#On sépare les stations en fonction de la pression industrielle (=nb d'usines aux alentours) qu'elles subissent
st["pression"] = pd.cut(st["usines_5km"], bins=[-1, 0, 5, 10000], labels=["0 usine", "1 à 5 usines", "plus de 5 usines"])
print(st["pression"].value_counts())

In [ ]:
evolution_par_pression = st.groupby("pression", observed=False)["evolution_m"].describe().round(2)
print(evolution_par_pression)

def fig_evolution_par_pression_boxplot():
    """Boîtes à moustaches de l'évolution selon le nombre d'usines à moins de 5 km."""
    st.boxplot(column="evolution_m", by="pression", figsize=(10, 6))
    plt.suptitle("")
    plt.title("Évolution des nappes selon la pression industrielle (rayon 5 km)")
    plt.xlabel("Nombre d'usines à moins de 5 km")
    plt.ylabel("Évolution (m)")
    plt.show()

if AFFICHER:
    fig_evolution_par_pression_boxplot()

**Interprétation.** Les trois boîtes sont centrées autour de zéro et ont des tailles comparables. Les médianes sont de +0,04 m (0 usine), +0,01 m (1 à 5 usines) et +0,14 m (plus de 5 usines) : des écarts de l'ordre du centimètre ou de la dizaine de centimètres, négligeables. La station typique évolue de la même façon, qu'elle ait 0 ou 15 usines autour d'elle. Le groupe le plus industriel a même la médiane la plus élevée.

La dispersion centrale ne diffère pas non plus : l'écart interquartile vaut 0,56 m sans usine, 0,29 m pour 1 à 5 usines et 0,33 m pour plus de 5 usines. Dans le groupe le plus industriel, les trois quarts des stations ont une évolution supérieure à −0,09 m.

La moyenne du groupe « plus de 5 usines » (−0,15 m) est pourtant la plus basse, alors que sa médiane est la plus haute. Cet écart vient de deux valeurs extrêmes, Bondues (−3,57 m) et Nieppe (−1,76 m), qui tirent la moyenne vers le bas et gonflent l'écart-type (0,94 m contre environ 0,6 m dans les autres groupes). C'est pourquoi on commente la médiane : la moyenne laisserait croire à un effet de l'industrie dû à seulement deux stations.

Les autres valeurs isolées sont Villers-en-Cauchies (−1,16 m), une baisse sans aucune usine à proximité, Lille (+0,81 m), une hausse malgré une forte présence industrielle, et Limont-Fontaine (+2,72 m), dont les données sont atypiques.

Le groupe « 0 usine » ne compte que 6 stations : ses statistiques sont à interpréter avec prudence.

In [ ]:
for rayon_km in [2, 5, 10]:
    print(f"Spearman évolution / usines à {rayon_km} km :", round(st["evolution_m"].corr(st[f"usines_{rayon_km}km"], method="spearman"), 2))

**Interprétation.** On utilise la corrélation de Spearman : elle mesure si « plus d'usines » va avec « plus de baisse ».

Elle vaut 0,25 à 2 km, 0,15 à 5 km et 0,17 à 10 km. Ces valeurs sont faibles et positives : on ne trouve aucune relation négative entre le nombre d'usines et l'évolution des nappes. Le résultat est stable quel que soit le rayon, ce qui le rend robuste.

In [ ]:
def fig_carte_evolutions():
    """Carte des stations colorées selon leur évolution, avec les usines en noir."""
    fig, ax = plt.subplots(figsize=(12, 10))
    usines.to_crs(epsg=3857).plot(ax=ax, color="black", markersize=5, alpha=0.4, label="Usines", zorder=2)
    st.to_crs(epsg=3857).plot(ax=ax, column="evolution_m", cmap="RdBu", norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), markersize=80, edgecolor="black", legend=True, legend_kwds={"label": "Évolution relative (m)"}, zorder=3)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap, zorder=1)
    ax.set_axis_off()
    plt.legend(loc="upper left")
    plt.title("Évolution des nappes entre 2015-2017 et 2023-2025 et localisation des usines")
    plt.show()

if AFFICHER:
    fig_carte_evolutions()

**Interprétation.** Les stations en baisse (rouge) et en hausse (bleu) sont réparties dans tout le département, sans concentration particulière autour des pôles industriels. Dans la métropole lilloise, où les usines sont les plus denses, la plupart des stations sont stables ou en hausse. Bondues, au nord de Lille, fait exception.

## 5.3 Ce qui distingue les stations en baisse

Puisque l'industrie n'explique pas les différences entre stations dans notre échantillon, on regarde les caractéristiques des stations extrêmes, puis l'effet de la géologie et de la profondeur.

In [ ]:
colonnes = ["code_bss", "nom_commune", "evolution_m", "usines_5km", "classe_profondeur", "famille_geologique"]
stations_extremes = pd.concat([st.sort_values("evolution_m")[colonnes].head(8), st.sort_values("evolution_m")[colonnes].tail(5)])
print("Plus fortes baisses")
print(st.sort_values("evolution_m")[colonnes].head(8).to_string(index=False))
print("\nPlus fortes hausses")
print(st.sort_values("evolution_m")[colonnes].tail(5).to_string(index=False))

**Interprétation.** Les deux plus fortes baisses sont entourées de nombreuses usines (Bondues : 21, Nieppe : 11), mais Villers-en-Cauchies (−1,16 m) et Steenbecque n'en ont aucune. À l'inverse, Lille, avec 22 usines, fait partie des plus fortes hausses (+0,81 m).

Bondues et Lille ont presque le même nombre d'usines mais des évolutions opposées. Ce qui les distingue est le type de nappe : Bondues est une nappe profonde dans le calcaire, Lille une nappe superficielle dans la craie.

In [ ]:
print(pd.crosstab(st["famille_geologique"], st["classe_profondeur"], margins=True))
print(st.groupby("famille_geologique")["evolution_m"].describe().round(2))
print(st.groupby("classe_profondeur", observed=False)["evolution_m"].describe().round(2))

def fig_evolution_geologie_profondeur():
    """Boîtes à moustaches de l'évolution selon la géologie et selon la profondeur."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    st.boxplot(column="evolution_m", by="famille_geologique", ax=axes[0])
    st.boxplot(column="evolution_m", by="classe_profondeur", ax=axes[1])
    axes[0].set_title("Selon la géologie")
    axes[1].set_title("Selon la profondeur")
    axes[0].set_xlabel("")
    axes[1].set_xlabel("")
    axes[0].set_ylabel("Évolution (m)")
    plt.suptitle("Évolution des nappes selon leurs caractéristiques")
    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_evolution_geologie_profondeur()

**Interprétation : la géologie.** C'est le facteur qui différencie le plus les nappes.

- **Craie** (24 stations) : nappes stables ou en légère hausse (médiane +0,12 m), avec peu de dispersion.
- **Sable** (5 stations) : seul groupe en baisse nette (médiane −0,34 m), dont Nieppe (−1,76 m). Résultat fragile vu le faible effectif.
- **Calcaire** (7 stations) : médiane positive (+0,23 m), mais le groupe contient les deux valeurs les plus extrêmes du département, Bondues (−3,57 m) et Limont-Fontaine (+2,72 m).
- **Autre** (14 stations) : géologie non renseignée, groupe non interprétable.

L'écart entre les médianes atteint 0,57 m selon la géologie, contre 0,13 m selon la pression industrielle : le type de nappe compte davantage que le nombre d'usines.

**Interprétation : la profondeur.** Les médianes sont toutes proches de zéro : la profondeur ne change pas l'évolution typique. En revanche, plus la nappe est profonde, plus les évolutions sont dispersées (écart-type de 0,33 m en surface contre 1,84 m en profondeur). Les nappes superficielles sont les plus stables, contrairement à l'hypothèse de départ.

**Attention :** 6 des 7 stations profondes sont aussi calcaires. On ne peut donc pas séparer l'effet de la profondeur de celui de la géologie.

## 5.4 Deuxième indicateur de pression : les volumes pompés

Le nombre d'usines est un indicateur imparfait : une petite usine et une grande usine comptent pareil, et une usine ne pompe pas forcément dans la nappe. Pour vérifier que l'absence de lien ne vient pas de ce choix d'indicateur, on utilise les volumes réellement prélevés.

La BNPE (Banque Nationale des Prélèvements quantitatifs en Eau) recense, pour chaque ouvrage (forage, puits, prise d'eau) et chaque année, le volume prélevé et son usage.

In [ ]:
# On télécharge les données sur les prélèvements (un fichier en cache par département)
url_bnpe = "https://hubeau.eaufrance.fr/api/v1/prelevements/chroniques"

def telecharger_toutes_pages(url, params_api):
    """Récupère toutes les pages d'une réponse Hub'eau en suivant le lien "next" (requete_hubeau réessaie en cas de coupure)."""
    lignes_api = []
    while url is not None:
        data = requete_hubeau(params_api, url)
        lignes_api.extend(data["data"])
        url = data.get("next")
        params_api = None   #Les paramètres sont déjà intégrés dans l'url de la page "next"
        time.sleep(0.5)     #On attend un peu pour éviter de se faire rejeter par le serveur
    return lignes_api

morceaux = []
for dep in DEPARTEMENTS:
    chemin_bnpe = f"../../data/prelevements_{dep}.csv"
    if not os.path.exists(chemin_bnpe):
        prelevements_dep = []
        #Comme pour le téléchargement des chroniques, les données de prélèvement sont trop lourdes à télécharger d'un seul coup : on les télécharge année après année.
        for annee in range(2015, 2026):
            prelevements_dep.extend(telecharger_toutes_pages(url_bnpe, {"code_departement": dep, "annee": annee, "size": 2000, "format": "json"}))
            print(f"Département {dep}, année {annee} terminée")
        pd.DataFrame(prelevements_dep).to_csv(chemin_bnpe, index=False, encoding="utf-8")
        print("Téléchargement réussi.")
    morceaux.append(pd.read_csv(chemin_bnpe, encoding="utf-8"))
prelevements = pd.concat(morceaux, ignore_index=True)

In [ ]:
url_ouvrages = "https://hubeau.eaufrance.fr/api/v1/prelevements/referentiel/ouvrages"

morceaux = []
for dep in DEPARTEMENTS:
    chemin_ouvrages = f"../../data/ouvrages_{dep}.csv"
    if not os.path.exists(chemin_ouvrages):
        pd.DataFrame(telecharger_toutes_pages(url_ouvrages, {"code_departement": dep, "size": 5000, "format": "json"})).to_csv(chemin_ouvrages, index=False, encoding="utf-8")
    morceaux.append(pd.read_csv(chemin_ouvrages, encoding="utf-8"))
ouvrages = pd.concat(morceaux, ignore_index=True).drop_duplicates(subset="code_ouvrage")

prelevements = prelevements.merge(ouvrages[["code_ouvrage", "libelle_type_milieu"]], on="code_ouvrage", how="left")
sout = prelevements[prelevements["libelle_type_milieu"].str.contains("souterrain", case=False, na=False)]
sout = gpd.GeoDataFrame(sout.drop(columns="geometry"), geometry=gpd.points_from_xy(sout["longitude"], sout["latitude"]), crs="EPSG:4326").to_crs(epsg=2154)

print(ouvrages["libelle_type_milieu"].value_counts())
print(len(prelevements), "prélèvements, dont", len(sout), "souterrains")

**Qualité des données.** Le nombre de prélèvements est stable d'une année à l'autre (500 à 614) : les données ne semblent pas tronquées. Elles s'arrêtent en 2023, en raison du délai de déclaration. Sur 776 ouvrages, 690 puisent dans une nappe et 86 en surface. On ne garde que les prélèvements souterrains, les seuls susceptibles de faire baisser une nappe.

In [ ]:
# Répartition de l'usage de l'eau pompée
noms_usages = {"AEP": "Eau potable", "IND": "Industrie", "IRR": "Irrigation"}
part = sout.groupby("code_usage")["volume"].sum().rename(index=noms_usages)
print((part / 1e6).round(1).sort_values(ascending=False))
print((100 * part / part.sum()).round(1).sort_values(ascending=False))

def fig_part_usages():
    """Part de chaque usage dans les volumes souterrains prélevés."""
    (100 * part / part.sum()).sort_values().plot(kind="barh", figsize=(8, 4), color="steelblue")
    plt.xlabel("Part des volumes souterrains prélevés (%)")
    plt.ylabel("")
    plt.title("Répartition des prélèvements en eau souterraine par usage (2015-2023)")
    plt.show()

if AFFICHER:
    fig_part_usages()

In [ ]:
# Évolution des prélèvements
par_an = sout.pivot_table(values="volume", index="annee", columns="code_usage", aggfunc="sum").rename(columns=noms_usages) / 1e6

comparaison = pd.DataFrame({"2015-2017": par_an.loc[2015:2017].mean(), "2021-2023": par_an.loc[2021:2023].mean()})
comparaison["évolution (%)"] = 100 * (comparaison["2021-2023"] / comparaison["2015-2017"] - 1)
print(comparaison.round(1))

def fig_prelevements_par_an():
    """Volumes souterrains prélevés chaque année, par usage."""
    par_an.plot(figsize=(10, 6), marker="o")
    plt.ylabel("Volume prélevé (millions de m³)")
    plt.xlabel("Année")
    plt.title("Évolution des prélèvements en eau souterraine par usage")
    plt.show()

if AFFICHER:
    fig_prelevements_par_an()

**Interprétation.** L'eau potable domine très largement les prélèvements en eau souterraine : 79,5 % des volumes entre 2015 et 2023, contre 18,2 % pour l'industrie et 2,3 % pour l'irrigation.

- **Eau potable** : environ 110 à 128 millions de m³ par an, en baisse d'environ 8 % entre 2015-2017 et 2021-2023.
- **Industrie** : stable autour de 25 à 29 millions de m³ par an (+3 % entre les deux périodes).
- **Irrigation** : volumes très faibles (2 à 7 millions de m³), plus élevés les années sèches (2020 et 2022).

L'industrie pompe donc peu dans les nappes du Nord, et de façon stable : il est cohérent que la présence d'usines ne se voie pas dans l'évolution des nappes. La corrélation légèrement positive ne s'explique pas non plus par une baisse des pompages industriels, puisque ceux-ci n'ont pas diminué.

In [ ]:
# Volume annuel moyen de chaque ouvrage souterrain, par usage
ouv = sout.groupby(["code_ouvrage", "code_usage"]).agg(volume=("volume", "mean"), geometry=("geometry", "first")).reset_index()
ouv = gpd.GeoDataFrame(ouv, geometry="geometry", crs=sout.crs).to_crs(epsg=3857)

def fig_carte_prelevements():
    """Carte des prélèvements souterrains, taille proportionnelle au volume annuel moyen."""
    couleurs = {"AEP": "royalblue", "IND": "black", "IRR": "green"}
    fig, ax = plt.subplots(figsize=(12, 10))
    for code, couleur in couleurs.items():
        sel = ouv[ouv["code_usage"] == code]
        sel.plot(ax=ax, color=couleur, markersize=np.sqrt(sel["volume"]) / 10, alpha=0.6, label=noms_usages[code], zorder=2)
    st[st["code_bss"].isin(["00143C0079/F1", "00147D0218/P1"])].to_crs(epsg=3857).plot(ax=ax, color="red", marker="*", markersize=300, label="Bondues / Lille", zorder=3)
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap, zorder=1)
    ax.set_axis_off()
    plt.legend(loc="upper left")
    plt.title("Prélèvements souterrains annuels moyens par usage (2015-2023)")
    plt.show()

if AFFICHER:
    fig_carte_prelevements()

**Interprétation.** La taille des cercles est proportionnelle au volume annuel moyen prélevé. Les prélèvements pour l'eau potable (bleu) sont les plus importants et se concentrent autour de la métropole lilloise, de Douai-Valenciennes et de l'Avesnois. Les prélèvements industriels (noir) sont dispersés et généralement plus petits. L'irrigation (vert) correspond à de nombreux petits prélèvements en Flandre intérieure et dans le Cambrésis.

In [ ]:
# Concentrons-nous sur les prélèvements autour des stations clés (stations du Nord : ignorées si elles ne font pas partie des départements étudiés)
codes_focus = ["00143C0079/F1", "00134X0127/PZ1", "00146B0007/F1", "00146X2045/PZF", "00146B0737/PZR2", "00147X3381/PZ6", "00147D0218/P1"]
lignes = []
for _, s in st[st["code_bss"].isin(codes_focus)].iterrows():
    proches = sout[sout.distance(s.geometry) <= 5000]
    nb_annees = max(proches["annee"].nunique(), 1)
    vol = proches.groupby("code_usage")["volume"].sum() / nb_annees / 1e6
    lignes.append({
        "code_bss": s["code_bss"],
        "commune": s["nom_commune"],
        "profondeur": s["classe_profondeur"],
        "geologie": s["famille_geologique"],
        "evolution_m": round(s["evolution_m"], 2),
        "eau_potable": round(vol.get("AEP", 0), 2),
        "industrie": round(vol.get("IND", 0), 2),
        "total": round(vol.sum(), 2)
    })
focus = pd.DataFrame(lignes, columns=["code_bss", "commune", "profondeur", "geologie", "evolution_m", "eau_potable", "industrie", "total"]).sort_values("evolution_m")
print(focus.to_string(index=False))

In [ ]:
def fig_bondues_lille():
    """Trajectoires annuelles de Bondues et des cinq piézomètres de Lille."""
    codes_duo = ["00143C0079/F1", "00146B0007/F1", "00146X2045/PZF", "00146B0737/PZR2", "00147X3381/PZ6", "00147D0218/P1"]
    codes_duo = [c for c in codes_duo if c in set(st["code_bss"])]   #Stations du Nord : ignorées si absentes
    plt.figure(figsize=(12, 6))
    for code in codes_duo:
        serie = annuel_ecart[annuel_ecart["code_bss"] == code]
        commune = st.loc[st["code_bss"] == code, "nom_commune"].iloc[0]
        style = {"color": "darkred", "linewidth": 3} if commune == "Bondues" else {"color": "steelblue", "alpha": 0.7}
        plt.plot(serie["annee"], serie["ecart"], marker="o", label=f"{commune} ({code})", **style)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Année")
    plt.ylabel("Écart au niveau moyen 2015-2017 (m)")
    plt.title("Bondues et les cinq piézomètres de Lille : des pompages comparables, des évolutions opposées")
    plt.legend(fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.show()

if AFFICHER:
    fig_bondues_lille()

**Interprétation.** Volumes annuels moyens prélevés à moins de 5 km, en millions de m³.

- **Nieppe** (−1,76 m) : quasiment aucun prélèvement (0,01 million de m³, uniquement pour l'irrigation). Sa baisse n'est pas due à un pompage local, encore moins à l'industrie. Elle s'inscrit dans la tendance des nappes sableuses.
- **Bondues** (−3,57 m) : 5,8 millions de m³ par an, dont 36 % pour l'industrie. Le pompage y est important, mais l'eau potable y pèse davantage que l'industrie.
- **Lille** : les cinq piézomètres sont tous en hausse (+0,16 à +0,81 m), alors qu'ils sont entourés de 3,3 à 7,0 millions de m³ de prélèvements annuels. Deux d'entre eux ont plus de prélèvements totaux que Bondues, et deux autres plus de prélèvements industriels (jusqu'à 4,5 millions de m³, plus du double de Bondues).

À pompage comparable, voire supérieur, les nappes de Lille montent alors que celle de Bondues baisse : ce n'est pas la quantité d'eau prélevée qui fait la différence.

Limite : la BNPE n'indique pas dans quelle nappe pompe chaque forage. Les prélèvements proches de Bondues ne visent pas forcément la même nappe que le piézomètre.

In [ ]:
moy = sout.groupby(["code_ouvrage", "code_usage"]).agg(volume=("volume", "mean"), geometry=("geometry", "first")).reset_index()
moy = gpd.GeoDataFrame(moy, geometry="geometry", crs=sout.crs)

st["vol_total_5km"] = st.geometry.apply(lambda g: moy.loc[moy.distance(g) <= 5000, "volume"].sum() / 1e6)
st["vol_ind_5km"] = st.geometry.apply(lambda g: moy.loc[(moy.distance(g) <= 5000) & (moy["code_usage"] == "IND"), "volume"].sum() / 1e6)

indicateurs = {
    "Nombre d'usines à 2 km": "usines_2km",
    "Nombre d'usines à 5 km": "usines_5km",
    "Nombre d'usines à 10 km": "usines_10km",
    "Volume total pompé à 5 km": "vol_total_5km",
    "Volume industriel pompé à 5 km": "vol_ind_5km"
}
tableau_corr = pd.DataFrame({
    "Indicateur de pression": list(indicateurs.keys()),
    "Spearman": [round(st["evolution_m"].corr(st[c], method="spearman"), 2) for c in indicateurs.values()]
})
print(tableau_corr.to_string(index=False))

**Interprétation.** Les cinq corrélations de Spearman sont faibles et positives (entre 0,15 et 0,27). Que la pression soit mesurée par le nombre d'usines ou par les volumes réellement pompés, aucune relation négative n'apparaît entre activité humaine et évolution des nappes. La conclusion ne dépend ni du rayon, ni de l'indicateur choisi.

La valeur légèrement plus élevée pour le volume industriel (0,27) ne signifie pas que pomper fait monter les nappes : les gros pompages industriels se situent surtout dans la métropole lilloise, au-dessus de la nappe de la craie qui se recharge vite. C'est alors sûrement un effet de la géologie. Mais une corrélation ne prouve pas une cause.

## 5.5 Conclusion de la partie 5

**L'activité industrielle ne fait pas baisser les nappes du Nord de façon générale.** Entre 2015-2017 et 2023-2025, la nappe typique est restée stable (médiane de +0,06 m), et les stations entourées d'usines n'ont pas évolué plus défavorablement que les autres. La corrélation de Spearman entre l'évolution des nappes et la pression humaine est faible et positive (entre 0,15 et 0,27), qu'on mesure cette pression par le nombre d'usines ou par les volumes pompés, et quel que soit le rayon.

**L'industrie est un préleveur minoritaire.** Elle ne représente que 18 % des prélèvements souterrains du département, contre 80 % pour l'eau potable, et ses volumes sont restés stables entre 2015 et 2023.

**La nature de la nappe semble compter davantage que l'industrie.** Les évolutions varient bien plus selon la géologie que selon la pression industrielle, et les cas extrêmes se concentrent dans les nappes profondes et calcaires. Les deux plus fortes baisses ne s'expliquent pas par l'industrie : il n'y a quasiment aucun prélèvement autour de Nieppe, et autour de Bondues l'eau potable pompe davantage que l'industrie. À l'inverse, les cinq piézomètres de Lille, entourés de prélèvements comparables ou supérieurs, sont tous en hausse.

**Réponse à la problématique.** Le risque pour les nappes ne vient pas de l'implantation d'usines en soi, mais du choix de la nappe sollicitée : une nappe profonde à recharge lente est bien plus vulnérable qu'une nappe superficielle qui se recharge rapidement. L'implantation de nouvelles usines très consommatrices d'eau devrait donc tenir compte de la nappe dans laquelle elles pomperont.

### Limites

- Un échantillon réduit : 50 piézomètres, avec de petits effectifs dans certains groupes.
- Des corrélations, qui ne prouvent pas une causalité. La géologie et la profondeur se recoupent en partie.
- Des données atypiques pour la station de Limont-Fontaine.

### Perspective

Il serait intéressant de reproduire cette analyse dans une région plus sèche, ou dont l'industrie est plus consommatrice d'eau, pour voir si les conclusions restent les mêmes.

In [ ]:
#Marqueur : cette étape est faite, les notebooks qui en dépendent ne la relanceront pas dans ce noyau
ETAPE_05 = True